# Phase 3 · Arch1 · Rhetorical roles

In [1]:
import os, sys, time, subprocess
from pathlib import Path

SESSION_START = time.monotonic()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub', 'tokenizers', 'cloudpickle', 'scikit-learn'], check=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Set HF_TOKEN in the environment before running

BASE = Path('/kaggle/input/datasets/kumaralpha1/a1-p3')
OUT = Path('/kaggle/working/phase3_arch1')


## Methods

In [2]:
import json, math, random, shutil, socket, zipfile, hashlib, gc, contextlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import cloudpickle
from datetime import timedelta
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from tokenizers import Tokenizer
from huggingface_hub import HfApi, hf_hub_download
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix

OUT.mkdir(parents=True, exist_ok=True)
C = dict(seed=42, micro=4, accum=4, infer_batch=8, target_tokens=256,
         neighbour_tokens=64, rank=16, alpha=32, bottleneck=64, prompts=20,
         head_lr=3e-4, dropout=0.1, weight_decay=0.01, warmup_fraction=0.1,
         patience=2, kd_temperature=2., kd_weight=0.5, checkpoint_every=100,
         session_hours=9., revision='main')
METHODS = {
    'full': dict(epochs=4, lr=2e-5),
    'partial': dict(epochs=5, lr=4e-5),
    'lora': dict(epochs=6, lr=2e-4),
    'lorma': dict(epochs=6, lr=2e-4),
    'adapter': dict(epochs=6, lr=3e-4),
    'ia3': dict(epochs=6, lr=5e-4),
    'prompt': dict(epochs=8, lr=3e-3),
    'distill': dict(epochs=5, lr=3e-5),
}

def atomic_save(obj, path):
    path = Path(path)
    torch.save(obj, str(path) + '.tmp')
    os.replace(str(path) + '.tmp', path)

def save_json(obj, path):
    path = Path(path)
    path.with_suffix('.tmp').write_text(json.dumps(obj, indent=2), encoding='utf-8')
    os.replace(path.with_suffix('.tmp'), path)

def save_array(path, array):
    path = Path(path)
    with open(str(path)+'.tmp', 'wb') as f:
        np.save(f, array)
    os.replace(str(path)+'.tmp', path)


## Official data and existing BPE

In [3]:
REPO = 'Exploration-Lab/CS-781-Capstone'
manifest_path = OUT / 'data_manifest.json'
data_revision = (json.loads(manifest_path.read_text())['revision'] if manifest_path.exists()
                 else HfApi(token=os.environ['HF_TOKEN']).dataset_info(REPO, revision=C['revision']).sha)
DATA_ROOT = OUT / 'dataset' / 'Phase_3_RR'
names = ['Train/CL_train.json', 'Train/CL_dev.json', 'Train/IT_train.json',
         'Train/IT_dev.json', 'Train/label_vocab.json', 'Test/CL_test.json', 'Test/IT_test.json']
for name in names:
    hf_hub_download(REPO, 'Phase_3_RR/' + name, repo_type='dataset', revision=data_revision,
                    token=os.environ['HF_TOKEN'], local_dir=OUT / 'dataset')

vocab = json.loads((DATA_ROOT / 'Train/label_vocab.json').read_text())
if isinstance(vocab, dict):
    vocab = vocab.get('label2id', vocab.get('id2label', vocab.get('labels', vocab)))
if isinstance(vocab, list):
    LABELS = vocab
elif all(str(k).isdigit() for k in vocab):
    LABELS = [vocab[k] for k in sorted(vocab, key=lambda x: int(x))]
else:
    LABELS = sorted(vocab, key=lambda x: int(vocab[x]))
label2id = {label: i for i, label in enumerate(LABELS)}

def read_docs(path, domain, labelled):
    obj = json.loads(Path(path).read_text())
    if isinstance(obj, dict) and 'data' in obj and isinstance(obj['data'], list):
        obj = obj['data']
    if isinstance(obj, dict) and 'id' in obj and 'text' in obj:
        obj = [obj]
    if isinstance(obj, dict):
        obj = [dict(v, id=k) if isinstance(v, dict) else dict(id=k, text=v) for k, v in obj.items()]
    docs = []
    for d in obj:
        texts = d['text']
        labels = d['labels'] if labelled else []
        labels = [label2id[y] if isinstance(y, str) and y in label2id else int(y) for y in labels]
        docs.append(dict(id=str(d['id']), text=texts, labels=labels, domain=domain))
    return docs

DOCS = {}
for split in ('train', 'dev', 'test'):
    folder = 'Test' if split == 'test' else 'Train'
    DOCS[split] = sum([read_docs(DATA_ROOT / folder / f'{d}_{split}.json', i, split != 'test')
                       for i, d in enumerate(('CL', 'IT'))], [])
DOCS['all'] = DOCS['train'] + DOCS['dev']

TOKENIZER = Tokenizer.from_file(str(BASE / 'tokenizer.json'))
TOKENIZER.no_padding()
TOKENIZER.no_truncation()
phase1 = torch.load(BASE / 'best.pt', map_location='cpu', weights_only=False)
MC = phase1['config'] if 'config' in phase1 else json.loads((BASE / 'config.json').read_text())
del phase1
PAD = TOKENIZER.token_to_id('<pad>')
EOS = TOKENIZER.token_to_id('<eos>')
PAD = MC.get('pad_id', 0) if PAD is None else PAD
EOS = MC.get('eos_id', 2) if EOS is None else EOS

def make_rows(docs):
    rows = []
    for di, doc in enumerate(docs):
        tokens = [e.ids for e in TOKENIZER.encode_batch(doc['text'], add_special_tokens=False)]
        for si, target in enumerate(tokens):
            limit = C['target_tokens']
            if len(target) > limit:
                target = target[:limit//2] + target[-(limit-limit//2):]
            target = target or [EOS]
            before = tokens[si-1][-C['neighbour_tokens']:] if si else []
            after = tokens[si+1][:C['neighbour_tokens']] if si+1 < len(tokens) else []
            ids = before + [EOS] + target + [EOS] + after
            pool = [0] * (len(before)+1) + [1] * len(target) + [0] * (len(after)+1)
            pos = si / max(1, len(tokens)-1)
            rows.append(dict(ids=ids, pool=pool, label=doc['labels'][si] if doc['labels'] else -100,
                             features=[pos, pos*pos, float(doc['domain']==0), float(doc['domain']==1)],
                             doc=di, sentence=si, index=len(rows)))
    return rows

ROWS = {s: make_rows(DOCS[s]) for s in ('train', 'dev', 'test', 'all')}
for split, rows in ROWS.items():
    atomic_save(rows, OUT / f'{split}_rows.pt')
save_json(LABELS, OUT / 'labels.json')
save_json(MC, OUT / 'model_config.json')
shutil.copy2(BASE / 'tokenizer.json', OUT / 'tokenizer.json')
save_json(dict(revision=data_revision, counts={s: len(ROWS[s]) for s in ROWS},
               documents={s: [d['id'] for d in DOCS[s]] for s in DOCS},
               tokenizer_sha256=hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest()), manifest_path)
save_json(dict(training=C, methods=METHODS, base=str(BASE)), OUT / 'run_config.json')


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

## Arch1 backbone

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['dim']
        self.heads = cfg['heads']
        self.head_dim = d // self.heads
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        inv = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos, sin = phase.cos().unsqueeze(1).to(x.dtype), phase.sin().unsqueeze(1).to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even*cos-odd*sin, even*sin+odd*cos), -1).flatten(-2)

    def forward(self, x, positions, mask):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        q, k = self.rope(q, positions), self.rope(k, positions)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, None, :],
                                          dropout_p=0., is_causal=False)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, d))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attention(cfg)
        self.gate, self.up = nn.Linear(d, f, bias=False), nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x, positions, mask):
        x = x + self.attn(self.norm1(x), positions, mask)
        z = self.norm2(x)
        return x + self.down(F.silu(self.gate(z)) * self.up(z))

class Backbone(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg['layers'])])
        self.norm = RMSNorm(cfg['dim'])


In [5]:
class LowRank(nn.Module):
    def __init__(self, base, multiplicative=False):
        super().__init__()
        self.base = base
        self.multiplicative = multiplicative
        self.scale = C['alpha'] / C['rank']
        width = base.out_features if multiplicative else base.in_features
        self.a = nn.Linear(width, C['rank'], bias=False)
        self.b = nn.Linear(C['rank'], base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))
        nn.init.zeros_(self.b.weight)

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * self.b(self.a(y if self.multiplicative else x))

class ScaleProjection(nn.Module):
    def __init__(self, base, qkv=False):
        super().__init__()
        self.base, self.qkv = base, qkv
        size = 2*base.out_features//3 if qkv else base.in_features
        self.scale = nn.Parameter(torch.ones(size))

    def forward(self, x):
        if not self.qkv:
            return self.base(x * self.scale)
        q, k, v = self.base(x).chunk(3, -1)
        sk, sv = self.scale.chunk(2)
        return torch.cat((q, k*sk, v*sv), -1)

class Adapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.down = nn.Linear(dim, C['bottleneck'])
        self.up = nn.Linear(C['bottleneck'], dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(self.norm(x))))

class RoleModel(nn.Module):
    def __init__(self, method):
        super().__init__()
        self.method = method
        self.backbone = Backbone(MC)
        payload = torch.load(BASE / 'best.pt', map_location='cpu', weights_only=False)
        state = payload['model']
        state = {k.removeprefix('module.'): v for k, v in state.items()}
        self.backbone.load_state_dict(state, strict=True)
        del payload, state
        d = MC['dim']
        if method == 'distill':
            chosen = np.linspace(0, MC['layers']-1, max(1, MC['layers']//2), dtype=int)
            self.backbone.blocks = nn.ModuleList([self.backbone.blocks[int(i)] for i in chosen])
        if method not in ('full', 'distill'):
            self.backbone.requires_grad_(False)
        if method == 'partial':
            for block in self.backbone.blocks[-6:]:
                block.requires_grad_(True)
            self.backbone.norm.requires_grad_(True)
        if method in ('lora', 'lorma'):
            for block in self.backbone.blocks:
                block.attn.qkv = LowRank(block.attn.qkv, method=='lorma')
                block.attn.out = LowRank(block.attn.out, method=='lorma')
                for name in ('gate', 'up', 'down'):
                    setattr(block, name, LowRank(getattr(block, name), method=='lorma'))
        if method == 'ia3':
            for block in self.backbone.blocks:
                block.attn.qkv = ScaleProjection(block.attn.qkv, qkv=True)
                block.down = ScaleProjection(block.down)
        self.adapters = nn.ModuleList([Adapter(d) for _ in self.backbone.blocks]) if method=='adapter' else None
        self.prompt = nn.Parameter(torch.randn(C['prompts'], d)*0.02) if method=='prompt' else None
        self.head = nn.Sequential(nn.LayerNorm(d+4), nn.Dropout(C['dropout']), nn.Linear(d+4, len(LABELS)))

    def forward(self, ids, mask, pool, features):
        x = self.backbone.embed(ids)
        if self.prompt is not None:
            p = self.prompt.unsqueeze(0).expand(x.size(0), -1, -1)
            x = torch.cat((p, x), 1)
            mask = torch.cat((torch.ones(p.shape[:2], device=x.device, dtype=torch.bool), mask), 1)
            pool = F.pad(pool, (p.size(1), 0))
        positions = torch.arange(x.size(1), device=x.device)[None, :]
        for i, block in enumerate(self.backbone.blocks):
            if self.training and torch.is_grad_enabled():
                x = checkpoint(block, x, positions, mask, use_reentrant=False)
            else:
                x = block(x, positions, mask)
            if self.adapters is not None:
                x = self.adapters[i](x)
        x = self.backbone.norm(x)
        pooled = (x.float()*pool.unsqueeze(-1)).sum(1) / pool.sum(1, keepdim=True)
        return self.head(torch.cat((pooled, features), -1))

    def delta(self):
        return {k: p.detach().cpu().clone() for k, p in self.named_parameters() if p.requires_grad}

    def restore(self, delta):
        params = dict(self.named_parameters())
        with torch.no_grad():
            for k, value in delta.items():
                params[k].copy_(value)


## Two-GPU workers

In [6]:
class Sentences(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]

def collate(rows):
    length = ((max(len(r['ids']) for r in rows)+7)//8)*8
    ids = torch.full((len(rows), length), PAD, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    pool = torch.zeros_like(ids, dtype=torch.float32)
    for i, row in enumerate(rows):
        n = len(row['ids'])
        ids[i, :n] = torch.tensor(row['ids'])
        mask[i, :n] = True
        pool[i, :n] = torch.tensor(row['pool'])
    return (ids, mask, pool, torch.tensor([r['features'] for r in rows], dtype=torch.float32),
            torch.tensor([r['label'] for r in rows]), torch.tensor([r['index'] for r in rows]))

def loader_for(rows, rank, world, training=False, epoch=0):
    ds = Sentences(rows)
    if training:
        sampler = DistributedSampler(ds, num_replicas=world, rank=rank, seed=C['seed'], shuffle=True)
        sampler.set_epoch(epoch)
    else:
        sampler = list(range(rank, len(rows), world))
    return DataLoader(ds, batch_size=C['micro'] if training else C['infer_batch'], sampler=sampler,
                      collate_fn=collate, num_workers=0, pin_memory=True,
                      generator=torch.Generator().manual_seed(C['seed']+epoch))

@torch.no_grad()
def predict(raw, rows, rank, world, device):
    raw.eval()
    logits = torch.zeros(len(rows), len(LABELS), device=device, dtype=torch.float32)
    for batch in loader_for(rows, rank, world):
        ids, mask, pool, features, _, index = [x.to(device, non_blocking=True) for x in batch]
        with torch.autocast('cuda', dtype=torch.float16):
            z = raw(ids, mask, pool, features)
        logits[index] = z.float()
    dist.all_reduce(logits, op=dist.ReduceOp.SUM)
    return logits.cpu().numpy()

def metrics(y, p, rows):
    pred = p.argmax(-1)
    pr, re, f1, _ = precision_recall_fscore_support(y, pred, labels=range(len(LABELS)), zero_division=0)
    result = dict(macro_f1=float(f1.mean()), precision=float(pr.mean()), recall=float(re.mean()),
                  per_label={name: dict(precision=float(pr[i]), recall=float(re[i]), f1=float(f1[i]))
                             for i, name in enumerate(LABELS)})
    for domain, feature in [('CL', 2), ('IT', 3)]:
        keep = np.array([r['features'][feature] == 1 for r in rows])
        result[domain] = float(f1_score(y[keep], pred[keep], labels=range(len(LABELS)), average='macro', zero_division=0))
    return result

def worker(rank, world, job):
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', rank=rank, world_size=world, timeout=timedelta(minutes=30))
    random.seed(C['seed'])
    np.random.seed(C['seed'])
    torch.manual_seed(C['seed'])
    torch.cuda.manual_seed(C['seed'])
    method, stage = job['method'], job['stage']
    folder = OUT / stage / method
    folder.mkdir(parents=True, exist_ok=True)
    raw = RoleModel(method).to(device)
    if job['action'] == 'predict':
        saved = torch.load(folder/'best.pt', map_location='cpu', weights_only=False)
        raw.restore(saved['delta'])
        for split in job['splits']:
            rows = torch.load(OUT/f'{split}_rows.pt', weights_only=False)
            z = predict(raw, rows, rank, world, device)
            if rank == 0:
                save_array(folder/f'{split}_logits.npy', z)
        dist.barrier()
        dist.destroy_process_group()
        return
    rows = torch.load(OUT/('train_rows.pt' if stage=='selection' else 'all_rows.pt'), weights_only=False)
    dev = torch.load(OUT/'dev_rows.pt', weights_only=False) if stage=='selection' else None
    y = np.array([r['label'] for r in rows])
    freq = np.bincount(y, minlength=len(LABELS))
    weights = np.sqrt(len(y) / (len(LABELS) * np.maximum(freq, 1)))
    weights = np.clip(weights, 0.5, 4.0)
    weights /= weights.mean()
    weights = torch.tensor(weights, device=device, dtype=torch.float32)
    soft = np.load(job['teacher_logits']) if method=='distill' else None
    trainable = [(n, p) for n, p in raw.named_parameters() if p.requires_grad]
    groups = []
    for is_head in (False, True):
        for decay in (False, True):
            params = [p for n, p in trainable if n.startswith('head.')==is_head and (p.ndim>=2)==decay]
            if params:
                groups.append(dict(params=params, lr=C['head_lr'] if is_head else METHODS[method]['lr'],
                                   initial_lr=C['head_lr'] if is_head else METHODS[method]['lr'],
                                   weight_decay=C['weight_decay'] if decay else 0.))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)
    scaler = torch.amp.GradScaler('cuda', init_scale=1024)
    epochs = job['epochs']
    batches = len(loader_for(rows, rank, world, True))
    updates = math.ceil(batches / C['accum'])
    total_steps = epochs * updates
    warmup = max(1, int(C['warmup_fraction'] * total_steps))
    epoch0, offset, step, best, best_epoch, stale = 0, 0, 0, -1., 1, 0
    loss_sum, seen, elapsed_before = 0., 0, 0.
    resume_path = folder/'last.pt'
    if resume_path.exists():
        state = torch.load(resume_path, map_location='cpu', weights_only=False)
        raw.restore(state['delta'])
        opt.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler'])
        epoch0, offset, step = state['epoch'], state['offset'], state['step']
        best, best_epoch, stale = state['best'], state['best_epoch'], state['stale']
        loss_sum, seen = state['loss_sum'][rank], state['seen'][rank]
        elapsed_before = state['elapsed']
        rng = state['rng'][rank]
        torch.set_rng_state(rng['cpu'])
        torch.cuda.set_rng_state(rng['cuda'], device)
        random.setstate(rng['python'])
        np.random.set_state(rng['numpy'])
        del state
    ddp = DDP(raw, device_ids=[rank], broadcast_buffers=False, gradient_as_bucket_view=True)
    started = time.monotonic()
    torch.cuda.reset_peak_memory_stats(device)

    def save_resume(epoch, offset):
        rng = dict(cpu=torch.get_rng_state(), cuda=torch.cuda.get_rng_state(device),
                   python=random.getstate(), numpy=np.random.get_state())
        local = dict(rng=rng, loss=loss_sum, seen=seen)
        gathered = [None]*world
        dist.all_gather_object(gathered, local)
        if rank == 0:
            atomic_save(dict(delta=raw.delta(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),
                             epoch=epoch, offset=offset, step=step, best=best, best_epoch=best_epoch,
                             stale=stale, rng=[g['rng'] for g in gathered],
                             loss_sum=[g['loss'] for g in gathered], seen=[g['seen'] for g in gathered],
                             elapsed=elapsed_before+time.monotonic()-started), resume_path)
        dist.barrier()

    epoch = epoch0 - 1
    end_epoch = epoch0 if stage=='selection' and stale >= C['patience'] else epochs
    for epoch in range(epoch0, end_epoch):
        raw.train()
        loader = loader_for(rows, rank, world, True, epoch)
        opt.zero_grad(set_to_none=True)
        for bi, batch in enumerate(loader):
            if bi < offset:
                continue
            ids, mask, pool, features, labels, indexes = [x.to(device, non_blocking=True) for x in batch]
            group_start = (bi//C['accum'])*C['accum']
            group_size = min(C['accum'], batches-group_start)
            sync = (bi+1)%C['accum']==0 or bi+1==batches
            with contextlib.nullcontext() if sync else ddp.no_sync():
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = ddp(ids, mask, pool, features)
                    loss = F.cross_entropy(logits.float(), labels, weight=weights)
                    if soft is not None:
                        teacher = torch.as_tensor(soft[indexes.cpu().numpy()], device=device)
                        t = C['kd_temperature']
                        kd = F.kl_div(F.log_softmax(logits.float()/t, -1), F.softmax(teacher/t, -1),
                                      reduction='batchmean') * t*t
                        loss = (1-C['kd_weight'])*loss + C['kd_weight']*kd
                scaler.scale(loss/group_size).backward()
            loss_sum += loss.item()*len(labels)
            seen += len(labels)
            if sync:
                factor = ((step+1)/warmup if step < warmup else
                          0.1+0.9*0.5*(1+math.cos(math.pi*(step-warmup)/max(1, total_steps-warmup))))
                for group in opt.param_groups:
                    group['lr'] = group['initial_lr']*factor
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(raw.parameters(), 1.)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                step += 1
                stop = torch.tensor(int(time.monotonic()-started > job['seconds']), device=device)
                dist.all_reduce(stop, op=dist.ReduceOp.MAX)
                if step%C['checkpoint_every']==0 or stop.item():
                    save_resume(epoch, bi+1)
                if stop.item():
                    dist.destroy_process_group()
                    return
        offset = 0
        totals = torch.tensor([loss_sum, seen], device=device, dtype=torch.float64)
        dist.all_reduce(totals)
        log = dict(epoch=epoch+1, loss=float(totals[0]/totals[1]), steps=step,
                   elapsed_seconds=elapsed_before+time.monotonic()-started)
        if stage == 'selection':
            z = predict(raw, dev, rank, world, device)
            report = metrics(np.array([r['label'] for r in dev]), z, dev)
            score = report['macro_f1']
            improved = score > best
            if improved:
                best, best_epoch, stale = score, epoch+1, 0
            else:
                stale += 1
            log['macro_f1'] = score
            if rank == 0 and improved:
                atomic_save(dict(delta=raw.delta(), method=method, epoch=best_epoch), folder/'best.pt')
                save_array(folder/'dev_logits.npy', z)
                save_json(report, folder/'metrics.json')
                save_array(folder/'confusion.npy', confusion_matrix([r['label'] for r in dev], z.argmax(-1), labels=range(len(LABELS))))
        elif rank == 0:
            atomic_save(dict(delta=raw.delta(), method=method, epoch=epoch+1), folder/'best.pt')
        if rank == 0:
            with (folder/'history.jsonl').open('a') as f:
                f.write(json.dumps(log)+'\n')
            print(f"{stage}/{method} | epoch {epoch+1}/{epochs} | loss {log['loss']:.4f}" +
                  (f" | macro-F1 {score:.4f}" if stage=='selection' else ''), flush=True)
        loss_sum, seen = 0., 0
        save_resume(epoch+1, 0)
        if stage=='selection' and stale >= C['patience']:
            break
    peak = torch.tensor(torch.cuda.max_memory_allocated(device)/2**30, device=device)
    dist.all_reduce(peak, op=dist.ReduceOp.MAX)
    if rank == 0:
        save_json(dict(method=method, best_epoch=best_epoch if stage=='selection' else epochs,
                       best_f1=best if stage=='selection' else None, completed_epochs=epoch+1,
                       parameters=sum(p.numel() for p in raw.parameters()),
                       trainable=sum(p.numel() for _, p in trainable), peak_gpu_gib=float(peak),
                       training_sentences=len(rows), world_size=world,
                       seconds=elapsed_before+time.monotonic()-started), folder/'done.json')
        resume_path.unlink(missing_ok=True)
    dist.barrier()
    dist.destroy_process_group()


In [7]:
def launch(job):
    remaining = C['session_hours']*3600 - (time.monotonic()-SESSION_START) - 180
    if remaining <= 0:
        raise RuntimeError('Session budget reached. Save the output folder and resume in a new session.')
    job = dict(job, seconds=remaining)
    with socket.socket() as sock:
        sock.bind(('127.0.0.1', 0))
        port = sock.getsockname()[1]
    payload = OUT / 'worker.pkl'
    with payload.open('wb') as f:
        cloudpickle.dump((worker, job), f)
    command = ('import cloudpickle,sys; '
               'f=open(sys.argv[1],"rb"); fn,job=cloudpickle.load(f); f.close(); '
               'fn(int(sys.argv[2]),2,job)')
    env = dict(os.environ, MASTER_ADDR='127.0.0.1', MASTER_PORT=str(port), OMP_NUM_THREADS='2')
    processes = [subprocess.Popen([sys.executable, '-u', '-c', command, str(payload), str(rank)], env=env)
                 for rank in range(2)]
    codes = [p.wait() for p in processes]
    payload.unlink(missing_ok=True)
    if any(codes):
        raise RuntimeError(f'DDP workers exited with codes {codes}.')
    if job['action']=='train' and not (OUT/job['stage']/job['method']/'done.json').exists():
        raise RuntimeError('Training checkpoint saved at session limit. Resume from the output folder.')

def fit(method, stage, epochs, teacher_logits=None):
    if not (OUT/stage/method/'done.json').exists():
        launch(dict(action='train', method=method, stage=stage, epochs=epochs, teacher_logits=teacher_logits))

def infer(method, stage, splits):
    missing = [s for s in splits if not (OUT/stage/method/f'{s}_logits.npy').exists()]
    if missing:
        launch(dict(action='predict', method=method, stage=stage, splits=missing))


## Select all eight methods

In [8]:
for method in METHODS:
    if method != 'distill':
        fit(method, 'selection', METHODS[method]['epochs'])

TEACHER = max((m for m in METHODS if m!='distill'),
              key=lambda m: json.loads((OUT/'selection'/m/'metrics.json').read_text())['macro_f1'])
save_json(dict(method=TEACHER), OUT/'teacher.json')
infer(TEACHER, 'selection', ['train'])
fit('distill', 'selection', METHODS['distill']['epochs'],
    str(OUT/'selection'/TEACHER/'train_logits.npy'))


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


selection/full | epoch 1/4 | loss 1.2269 | macro-F1 0.3521


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


selection/full | epoch 2/4 | loss 0.3161 | macro-F1 0.3602


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


selection/full | epoch 3/4 | loss 0.0839 | macro-F1 0.3537


... (69 outputs left out)

selection/distill | epoch 4/5 | loss 0.2391 | macro-F1 0.3830


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


selection/distill | epoch 5/5 | loss 0.2121 | macro-F1 0.3905


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


## Freeze ensemble choices before refitting

In [9]:
def probabilities(z, temperature=1.):
    z = z.astype(np.float64)/temperature
    z -= z.max(-1, keepdims=True)
    p = np.exp(z)
    return p/p.sum(-1, keepdims=True)

def transition_stats(docs):
    k = len(LABELS)
    start, trans = np.ones(k), np.ones((k, k))
    for d in docs:
        y = d['labels']
        if y:
            start[y[0]] += 1
        for a, b in zip(y, y[1:]):
            trans[a, b] += 1
    return np.log(start/start.sum()), np.log(trans/trans.sum(1, keepdims=True))

def decode(p, docs, transitions, strength):
    start, trans = transitions
    output, offset = [], 0
    for d in docs:
        n = len(d['text'])
        emission = np.log(np.clip(p[offset:offset+n], 1e-10, 1.))
        offset += n
        if not n:
            continue
        if strength == 0:
            output.extend(emission.argmax(-1).tolist())
            continue
        value, backs = emission[0]+strength*start, []
        for t in range(1, n):
            scores = value[:, None]+strength*trans
            backs.append(scores.argmax(0))
            value = emission[t]+scores.max(0)
        path = [int(value.argmax())]
        for back in reversed(backs):
            path.append(int(back[path[-1]]))
        output.extend(reversed(path))
    return np.asarray(output, dtype=int)

NAMES = list(METHODS)
DEV_Z = [np.load(OUT/'selection'/m/'dev_logits.npy') for m in NAMES]
DEV_Y = np.array([r['label'] for r in ROWS['dev']])
scores = np.array([json.loads((OUT/'selection'/m/'metrics.json').read_text())['macro_f1'] for m in NAMES])
weights_options = [np.ones(len(NAMES))/len(NAMES), np.maximum(scores, 1e-3)**2]
weights_options[1] /= weights_options[1].sum()
for count in (3, 5):
    keep = np.argsort(scores)[-count:]
    for power in (0, 2):
        w = np.zeros(len(NAMES))
        w[keep] = np.maximum(scores[keep], 1e-3)**power
        weights_options.append(w/w.sum())
transitions = transition_stats(DOCS['train'])
ensemble = dict(macro_f1=-1.)
for wi, weights in enumerate(weights_options):
    for temperature in (0.75, 1., 1.5):
        p = sum(w*probabilities(z, temperature) for w, z in zip(weights, DEV_Z))
        for strength in (0., 0.1, 0.25, 0.5):
            pred = decode(p, DOCS['dev'], transitions, strength)
            score = f1_score(DEV_Y, pred, labels=range(len(LABELS)), average='macro', zero_division=0)
            if score > ensemble['macro_f1']:
                ensemble = dict(macro_f1=float(score), weights=weights.tolist(), methods=NAMES,
                                temperature=temperature, transition_strength=strength)
save_json(ensemble, OUT/'ensemble_selection.json')


## Nine submissions before refitting

In [10]:
def write_submission(name, predicted, directory):
    result, offset = {}, 0
    for doc in DOCS['test']:
        n = len(doc['text'])
        result[doc['id']] = [LABELS[int(i)] for i in predicted[offset:offset+n]]
        offset += n
    if offset != len(predicted) or any(len(result[d['id']]) != len(d['text']) for d in DOCS['test']):
        raise ValueError('Submission sentence counts do not match the official test documents.')
    text = json.dumps(result, ensure_ascii=False)
    (directory/f'prediction_{name}.json').write_text(text, encoding='utf-8')
    with zipfile.ZipFile(directory/f'submission_A1_{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json', text)


def create_submissions(stage):
    directory = OUT/'submissions'/stage
    directory.mkdir(parents=True, exist_ok=True)
    logits = []
    for method in NAMES:
        infer(method, stage, ['test'])
        z = np.load(OUT/stage/method/'test_logits.npy')
        logits.append(z)
        write_submission(method, z.argmax(-1), directory)
    ensemble = json.loads((OUT/'ensemble_selection.json').read_text())
    p = sum(w*probabilities(z, ensemble['temperature'])
            for w, z in zip(ensemble['weights'], logits))
    labelled = DOCS['train'] if stage=='selection' else DOCS['all']
    predicted = decode(p, DOCS['test'], transition_stats(labelled), ensemble['transition_strength'])
    write_submission('ensemble', predicted, directory)
    bundle = OUT/f'Arch1_Phase39_submissions_{stage}.zip'
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
        for method in NAMES + ['ensemble']:
            path = directory/f'submission_A1_{method}.zip'
            z.write(path, path.name)
    from IPython.display import FileLink, display
    display(FileLink(str(bundle)))

create_submissions('selection')


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/p

/kaggle/working/phase3_arch1/Arch1_Phase3_9_submissions_selection.zip

## Refit every method on train + development

In [11]:
for method in NAMES:
    if method == 'distill':
        continue
    selected = json.loads((OUT/'selection'/method/'done.json').read_text())
    fit(method, 'refit', selected['best_epoch'])

infer(TEACHER, 'refit', ['all'])
selected = json.loads((OUT/'selection'/'distill'/'done.json').read_text())
fit('distill', 'refit', selected['best_epoch'], str(OUT/'refit'/TEACHER/'all_logits.npy'))


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


refit/full | epoch 1/4 | loss 1.2403
refit/full | epoch 2/4 | loss 0.3342
refit/full | epoch 3/4 | loss 0.0962
refit/full | epoch 4/4 | loss 0.0365


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


refit/partial | epoch 1/1 | loss 1.1886


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


refit/lora | epoch 1/1 | loss 1.2905


... (6 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


refit/prompt | epoch 1/7 | loss 1.8329
refit/prompt | epoch 2/7 | loss 1.3719
refit/prompt | epoch 3/7 | loss 1.2290
refit/prompt | epoch 4/7 | loss 1.1378
refit/prompt | epoch 5/7 | loss 1.0682
refit/prompt | epoch 6/7 | loss 1.0061
refit/prompt | epoch 7/7 | loss 0.9594


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


refit/distill | epoch 1/5 | loss 0.9812
refit/distill | epoch 2/5 | loss 0.4011
refit/distill | epoch 3/5 | loss 0.2894
refit/distill | epoch 4/5 | loss 0.2408
refit/distill | epoch 5/5 | loss 0.2140


## Nine refitted submissions

In [12]:
create_submissions('refit')

summary = []
for method in NAMES:
    selected = json.loads((OUT/'selection'/method/'done.json').read_text())
    refit = json.loads((OUT/'refit'/method/'done.json').read_text())
    summary.append(dict(method=method, dev_macro_f1=selected['best_f1'],
                        refit_epochs=refit['completed_epochs'], parameters=refit['parameters'],
                        trainable=refit['trainable'], refit_sentences=refit['training_sentences'],
                        peak_gpu_gib=refit['peak_gpu_gib'], seconds=refit['seconds'],
                        codabench_score=None))
save_json(summary, OUT/'results.json')


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
/usr/local/lib/p

/kaggle/working/phase3_arch1/Arch1_Phase3_9_submissions_refit.zip